# Phase 7 F1 — repetition-penalty sweep on the selected Epoch 3 checkpoint

This private Kaggle notebook selects a decoding policy for the already-trained F1 Epoch 3 checkpoint. It **does not train a model** and it **never reads the test set**.

## Scientific contract

- Checkpoint: selected F1 Epoch 3 checkpoint.
- Validation: the same frozen 732 clips and 64,137 NFD reference code points.
- Only changed variable: `repetition_penalty` in `1.00, 1.02, 1.05, 1.08, 1.10`.
- Fixed decoding: deterministic greedy decoding, one beam, batch 8, FP32, maximum length 448, explicit attention mask, `task="transcribe"`, automatic language detection, `forced_decoder_ids=None`, `no_repeat_ngram_size=0`, and the already-audited suppression policy.
- Fixed scoring: corpus code-point CER after NFD, outer trim, and repeated ordinary-space collapse. Spaces and IPA marks remain significant.

## Where to run

Run this notebook on **Kaggle with a GPU**. A T4 is sufficient. Local CPU decoding would be much slower.

Attach these private Kaggle Datasets:

1. The Phase 5 validation bundle (`phase5-validation-bundle`), either extracted or as `phase5_validation_bundle.zip`.
2. The F1 best checkpoint (`phase7_F1_best_checkpoint`), either extracted or as `phase7_F1_best_checkpoint_private.zip`.
3. Recommended: the prior Epoch 3 validation-audit ZIP. When present, this notebook verifies and reuses the saved `1.00` and `1.05` prediction files, then generates only `1.02`, `1.08`, and `1.10`. Without it, all five settings are generated.

Enable **Internet** so the pinned base processor can be downloaded. Keep every attached Dataset and the notebook private because the outputs contain references, predictions, and IDs.

> A repetition penalty acts inside autoregressive generation. It cannot be calculated by editing saved prediction text. Saved predictions can only be re-scored or reused for settings that were already generated.


In [ ]:
# Check the environment before using pip. Kaggle's CUDA-enabled PyTorch
# build is intentionally left untouched. This cell prints live pip output
# and fails within five minutes instead of hanging indefinitely.
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

REQUIRED = {
    "transformers": "4.57.1",
    "tokenizers": "0.22.2",
    "huggingface-hub": "0.36.2",
    "soundfile": "0.13.1",
    "safetensors": "0.8.0",
}

def installed_version(package_name):
    # Return None when a package is absent instead of failing the preflight.
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None

before = {name: installed_version(name) for name in REQUIRED}
missing_or_mismatched = {
    name: wanted for name, wanted in REQUIRED.items() if before[name] != wanted
}
print({"installed_before": before, "needs_install": missing_or_mismatched})

if missing_or_mismatched:
    command = [
        sys.executable, "-m", "pip", "install", "--no-input",
        "--disable-pip-version-check", "--retries", "2", "--timeout", "30",
        *[f"{name}=={wanted}" for name, wanted in missing_or_mismatched.items()],
    ]
    print("Running:", " ".join(command))
    try:
        subprocess.run(command, check=True, timeout=300)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(
            "Package installation exceeded five minutes. Kaggle Internet or its package "
            "index is unavailable. Enable Internet, restart the session, and rerun."
        ) from exc
else:
    print("Exact pinned package versions are already installed; pip was skipped.")

after = {name: installed_version(name) for name in REQUIRED}
assert after == REQUIRED, f"Package-version verification failed: {after}"
print({"package_preflight": "PASS", "installed_after": after})


## 1. Imports, constants, and runtime checks

This block records the environment, fixes the declared experiment constants, and refuses to continue without a CUDA GPU. The test split is not an accepted input.


In [ ]:
# Standard-library imports support hashing, safe ZIP extraction, scoring,
# timing, and private result packaging.
import copy
import csv
import gc
import hashlib
import json
import os
import platform
import re
import time
import unicodedata
import warnings
import zipfile
from importlib.metadata import version
from pathlib import Path

# Third-party imports provide audio I/O, tables, plots, progress bars, GPU
# inference, and the pinned Whisper implementation.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
import transformers
from IPython.display import FileLink, display
from tqdm.auto import tqdm
from transformers import GenerationConfig, WhisperForConditionalGeneration, WhisperProcessor

# Avoid tokenizer worker messages. Text decoding below uses batch_decode and
# does not use the slower encode-then-pad pattern.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL_ID = "neurlang/ipa-whisper-base"
MODEL_REVISION = "949f7c4e4ee374b5202bc5886f6e4552e5ae2371"
EXPECTED_CLIPS = 732
EXPECTED_REFERENCE_CODEPOINTS = 64137
EOS_ID = 50257
MAX_LENGTH = 448
VALIDATION_BATCH = 8
PENALTIES = [1.00, 1.02, 1.05, 1.08, 1.10]

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/phase7_f1_repetition_penalty_sweep_private")
EXTRACT_ROOT = WORK_ROOT / "_extracted_inputs"
OUTPUT_ROOT = WORK_ROOT / "results"
PLOTS_ROOT = WORK_ROOT / "plots"
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PLOTS_ROOT.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is active. In Kaggle, open Settings, choose GPU as the "
        "accelerator, save, restart the session, and rerun from the top."
    )

print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "soundfile": version("soundfile"),
    "gpu": torch.cuda.get_device_name(0),
    "cuda_runtime": torch.version.cuda,
    "penalties": PENALTIES,
})


## 2. Discover and safely materialize the private inputs

Kaggle may mount a Dataset as extracted files or may expose the uploaded ZIP. This block supports both forms, validates ZIP paths before extraction, and rejects ambiguous inputs rather than silently selecting the wrong run.


In [ ]:
# Hash files in streaming blocks so large checkpoint files do not fill RAM.
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Resolve a relative path and prove that it remains inside its declared root.
def safe_path(root: Path, relative: str) -> Path:
    resolved = (root / relative).resolve()
    resolved.relative_to(root.resolve())
    return resolved


# Extract an archive only after checking every destination against ZIP-slip.
def safe_extract(archive_path: Path, destination: Path) -> Path:
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in tqdm(archive.infolist(), desc=f"Checking {archive_path.name}", unit="file"):
            (destination / member.filename).resolve().relative_to(destination.resolve())
        for member in tqdm(archive.infolist(), desc=f"Extracting {archive_path.name}", unit="file"):
            archive.extract(member, destination)
    return destination


# Read a bundle marker from a ZIP without extracting it.
def zip_bundle_types(path: Path) -> set[str]:
    found = set()
    try:
        with zipfile.ZipFile(path) as archive:
            for name in archive.namelist():
                if name.endswith("bundle_record.json"):
                    found.add(json.loads(archive.read(name))["bundle_type"])
    except (OSError, KeyError, json.JSONDecodeError, zipfile.BadZipFile):
        pass
    return found


# Locate exactly one frozen validation bundle, preferring an extracted Dataset
# over a duplicate ZIP in that same Dataset mount.
validation_direct = []
for marker in INPUT_ROOT.rglob("bundle_record.json"):
    try:
        if json.loads(marker.read_text())["bundle_type"] == "phase5_validation_baseline":
            validation_direct.append(marker.parent)
    except (OSError, KeyError, json.JSONDecodeError):
        pass
validation_zips = [
    path for path in INPUT_ROOT.rglob("*.zip")
    if "phase5_validation_baseline" in zip_bundle_types(path)
]

if validation_direct:
    if len(validation_direct) != 1:
        raise RuntimeError(f"Expected one extracted validation bundle, found: {validation_direct}")
    validation_root = validation_direct[0]
    validation_source = validation_root
else:
    if len(validation_zips) != 1:
        raise RuntimeError(f"Expected one validation-bundle ZIP, found: {validation_zips}")
    extracted = safe_extract(validation_zips[0], EXTRACT_ROOT / "validation")
    markers = [
        marker for marker in extracted.rglob("bundle_record.json")
        if json.loads(marker.read_text()).get("bundle_type") == "phase5_validation_baseline"
    ]
    if len(markers) != 1:
        raise RuntimeError(f"Expected one validation marker after extraction, found: {markers}")
    validation_root = markers[0].parent
    validation_source = validation_zips[0]

# Locate exactly one model directory. If only a private checkpoint ZIP is
# mounted, extract it and then locate the directory containing safetensors.
checkpoint_models = [
    path.parent for path in INPUT_ROOT.rglob("model.safetensors")
    if (path.parent / "config.json").is_file()
]
checkpoint_zips = []
for path in INPUT_ROOT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(path) as archive:
            names = set(archive.namelist())
        if any(name.endswith("model/model.safetensors") for name in names) and any(
            name.endswith("model/config.json") for name in names
        ):
            checkpoint_zips.append(path)
    except (OSError, zipfile.BadZipFile):
        pass

if checkpoint_models:
    if len(checkpoint_models) != 1:
        raise RuntimeError(f"Expected one extracted checkpoint model, found: {checkpoint_models}")
    checkpoint_model_root = checkpoint_models[0]
    checkpoint_source = checkpoint_model_root
else:
    if len(checkpoint_zips) != 1:
        raise RuntimeError(f"Expected one checkpoint ZIP, found: {checkpoint_zips}")
    extracted = safe_extract(checkpoint_zips[0], EXTRACT_ROOT / "checkpoint")
    checkpoint_models = [
        path.parent for path in extracted.rglob("model.safetensors")
        if (path.parent / "config.json").is_file()
    ]
    if len(checkpoint_models) != 1:
        raise RuntimeError(f"Expected one checkpoint model after extraction, found: {checkpoint_models}")
    checkpoint_model_root = checkpoint_models[0]
    checkpoint_source = checkpoint_zips[0]

# The prior audit is optional. Its prediction files save two full validation
# passes when it is attached and verifies continuity with the earlier audit.
audit_roots = [
    path.parent for path in INPUT_ROOT.rglob("corrected_epoch_03_predictions_private.jsonl")
]
audit_zips = []
for path in INPUT_ROOT.rglob("*.zip"):
    try:
        with zipfile.ZipFile(path) as archive:
            names = set(archive.namelist())
        if any(name.endswith("corrected_epoch_03_predictions_private.jsonl") for name in names):
            audit_zips.append(path)
    except (OSError, zipfile.BadZipFile):
        pass

audit_root = None
audit_source = None
if audit_roots:
    if len(audit_roots) != 1:
        raise RuntimeError(f"Expected at most one extracted audit, found: {audit_roots}")
    audit_root, audit_source = audit_roots[0], audit_roots[0]
elif audit_zips:
    if len(audit_zips) != 1:
        raise RuntimeError(f"Expected at most one audit ZIP, found: {audit_zips}")
    extracted = safe_extract(audit_zips[0], EXTRACT_ROOT / "audit")
    matches = list(extracted.rglob("corrected_epoch_03_predictions_private.jsonl"))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one saved baseline prediction file, found: {matches}")
    audit_root, audit_source = matches[0].parent, audit_zips[0]

print({
    "validation_input": str(validation_source),
    "checkpoint_input": str(checkpoint_source),
    "prior_audit_input": str(audit_source) if audit_source else None,
    "reuse_saved_predictions": audit_root is not None,
})


## 3. Verify the frozen validation contract and checkpoint

This block verifies available bundle checksums, confirms that the bundle is validation-only, checks all denominators, and ensures the checkpoint is a Whisper model. It never searches for or opens a test manifest.


In [ ]:
# Verify the validation bundle's declaration before reading its manifest.
bundle_record = json.loads((validation_root / "bundle_record.json").read_text())
assert bundle_record["bundle_type"] == "phase5_validation_baseline"
assert bundle_record["evaluation_partition"] == "validation"
assert bundle_record["clips"] == EXPECTED_CLIPS
assert bundle_record["reference_codepoints"] == EXPECTED_REFERENCE_CODEPOINTS
assert bundle_record.get("contains_train_audio") is False
assert bundle_record.get("contains_test_audio") is False

# Verify every checksum supplied by the private bundle.
checksum_path = validation_root / "SHA256SUMS.txt"
if checksum_path.is_file():
    checksum_entries = [
        line.split("  ", 1) for line in checksum_path.read_text().splitlines() if line.strip()
    ]
    for expected_hash, relative_path in tqdm(
        checksum_entries, desc="Verifying validation checksums", unit="file"
    ):
        actual_hash = sha256(safe_path(validation_root, relative_path))
        assert actual_hash == expected_hash, f"Checksum mismatch: {relative_path}"

# Read the frozen validation rows and re-establish exact denominators.
with (validation_root / "validation_manifest.csv").open(newline="", encoding="utf-8") as stream:
    validation_rows = list(csv.DictReader(stream))

def normalize_ipa(value: str) -> str:
    # Apply only the approved NFD/outer-trim/repeated-U+0020 policy.
    return re.sub(r" {2,}", " ", unicodedata.normalize("NFD", str(value)).strip())

assert len(validation_rows) == EXPECTED_CLIPS
assert len({row["sample_id"] for row in validation_rows}) == EXPECTED_CLIPS
assert sum(len(normalize_ipa(row["reference"])) for row in validation_rows) == EXPECTED_REFERENCE_CODEPOINTS
assert all(0.5 <= float(row["duration_seconds"]) < 30 for row in validation_rows)

# Verify that the local checkpoint has the expected Whisper architecture.
checkpoint_config = json.loads((checkpoint_model_root / "config.json").read_text())
assert checkpoint_config.get("model_type") == "whisper"
assert (checkpoint_model_root / "model.safetensors").stat().st_size > 100_000_000

print({
    "validation_clips": len(validation_rows),
    "reference_codepoints": EXPECTED_REFERENCE_CODEPOINTS,
    "validation_hours": sum(float(row["duration_seconds"]) for row in validation_rows) / 3600,
    "checkpoint_model_dir": str(checkpoint_model_root),
    "checkpoint_weights_mib": round(
        (checkpoint_model_root / "model.safetensors").stat().st_size / 2**20, 1
    ),
    "test_data_used": False,
})


## 4. Load the pinned processor and selected F1 checkpoint

The processor is downloaded from the pinned public model revision. Model weights come only from the attached F1 checkpoint. The effective generation configuration is printed and saved before decoding.


In [ ]:
# Load processor metadata from the exact pinned base revision and load the
# selected model weights from the private Epoch 3 checkpoint.
device = torch.device("cuda")
processor = WhisperProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = WhisperForConditionalGeneration.from_pretrained(
    checkpoint_model_root,
    dtype=torch.float32,
    local_files_only=True,
).to(device)
model.eval()
model.config.use_cache = True

# Start from the frozen validation bundle's generation config, then explicitly
# reproduce the corrected Phase 7 audit policy. Only repetition_penalty changes.
generation_config = GenerationConfig.from_dict(
    json.loads((validation_root / "generation_config.json").read_text())
)
generation_config.max_length = MAX_LENGTH
generation_config.max_new_tokens = None
generation_config.do_sample = False
generation_config.num_beams = 1
generation_config.num_return_sequences = 1
generation_config.repetition_penalty = 1.0
generation_config.no_repeat_ngram_size = 0
generation_config.suppress_tokens = []
generation_config.begin_suppress_tokens = [220, EOS_ID]
generation_config.forced_decoder_ids = None
generation_config.language = None
generation_config.task = "transcribe"
generation_config.return_timestamps = False
generation_config.return_dict_in_generate = True
generation_config.output_scores = False
generation_config.pad_token_id = EOS_ID
generation_config.eos_token_id = EOS_ID

assert generation_config.eos_token_id == EOS_ID
assert generation_config.decoder_start_token_id == 50258
assert generation_config.forced_decoder_ids is None
assert generation_config.language is None
assert generation_config.task == "transcribe"
assert generation_config.suppress_tokens == []
assert generation_config.begin_suppress_tokens == [220, EOS_ID]
assert generation_config.no_repeat_ngram_size == 0

effective_policy = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "checkpoint": "F1 Epoch 3 selected checkpoint",
    "validation_clips": EXPECTED_CLIPS,
    "reference_codepoints": EXPECTED_REFERENCE_CODEPOINTS,
    "inference_dtype": str(next(model.parameters()).dtype),
    "validation_batch": VALIDATION_BATCH,
    "attention_mask_supplied": True,
    "max_length": generation_config.max_length,
    "do_sample": generation_config.do_sample,
    "num_beams": generation_config.num_beams,
    "task": generation_config.task,
    "language": generation_config.language,
    "forced_decoder_ids": generation_config.forced_decoder_ids,
    "suppress_tokens": generation_config.suppress_tokens,
    "begin_suppress_tokens": generation_config.begin_suppress_tokens,
    "no_repeat_ngram_size": generation_config.no_repeat_ngram_size,
    "penalties": PENALTIES,
    "test_data_used": False,
}
(OUTPUT_ROOT / "effective_generation_policy.json").write_text(
    json.dumps(effective_policy, indent=2, ensure_ascii=False) + "\n"
)
print(json.dumps(effective_policy, indent=2, ensure_ascii=False))

# The configuration was printed and asserted above. Lower the Transformers
# logger only after the audit so known informational generation messages do not
# obscure the notebook's own tqdm progress bars.
transformers.utils.logging.set_verbosity_error()


## 5. Exact scoring, audio, and batch-feature helpers

The edit-distance backtrace matches the established project implementation. Feature extraction forces Whisper's fixed 30-second, 80×3000 input and returns the explicit encoder attention mask required by the corrected audit.


In [ ]:
# Return one deterministic optimal decomposition into substitutions, deletions,
# and insertions over Unicode code points.
def edit_counts(reference: str, hypothesis: str) -> tuple[int, int, int]:
    ref, hyp = list(reference), list(hypothesis)
    distance = [[0] * (len(hyp) + 1) for _ in range(len(ref) + 1)]
    for i in range(len(ref) + 1):
        distance[i][0] = i
    for j in range(len(hyp) + 1):
        distance[0][j] = j
    for i in range(1, len(ref) + 1):
        for j in range(1, len(hyp) + 1):
            distance[i][j] = min(
                distance[i - 1][j] + 1,
                distance[i][j - 1] + 1,
                distance[i - 1][j - 1] + (ref[i - 1] != hyp[j - 1]),
            )
    substitutions = deletions = insertions = 0
    i, j = len(ref), len(hyp)
    while i or j:
        if i and j and ref[i - 1] == hyp[j - 1] and distance[i][j] == distance[i - 1][j - 1]:
            i, j = i - 1, j - 1
        elif i and j and distance[i][j] == distance[i - 1][j - 1] + 1:
            substitutions += 1
            i, j = i - 1, j - 1
        elif i and distance[i][j] == distance[i - 1][j] + 1:
            deletions += 1
            i -= 1
        else:
            insertions += 1
            j -= 1
    return substitutions, deletions, insertions


# Read one original 16 kHz mono clip without applying an audio transform.
def read_audio(row: dict) -> np.ndarray:
    audio_path = safe_path(validation_root, row["audio_file"])
    audio, sample_rate = sf.read(audio_path, dtype="float32", always_2d=True)
    assert sample_rate == 16000 and audio.shape[1] == 1
    assert 0.5 <= len(audio) / sample_rate < 30
    assert np.isfinite(audio).all()
    return audio[:, 0]


# Produce FP32 log-Mel features and the feature-axis attention mask. The
# defensive crop/pad handles harmless feature-extractor patch differences.
def make_batch(rows: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    audios = [read_audio(row) for row in rows]
    packed = processor.feature_extractor(
        audios,
        sampling_rate=16000,
        return_tensors="pt",
        return_attention_mask=True,
        padding="max_length",
        truncation=True,
        max_length=480000,
    )
    features = packed.input_features
    attention_mask = packed.attention_mask
    if features.ndim == 2:
        features = features.unsqueeze(0)
    if attention_mask.ndim == 1:
        attention_mask = attention_mask.unsqueeze(0)
    if features.shape[-1] > 3000:
        features = features[..., :3000]
    elif features.shape[-1] < 3000:
        features = F.pad(features, (0, 3000 - features.shape[-1]))
    if attention_mask.shape[-1] > 3000:
        attention_mask = attention_mask[..., :3000]
    elif attention_mask.shape[-1] < 3000:
        attention_mask = F.pad(attention_mask, (0, 3000 - attention_mask.shape[-1]))
    assert tuple(features.shape[1:]) == (80, 3000)
    assert tuple(attention_mask.shape) == (len(rows), 3000)
    assert torch.isfinite(features).all().item()
    return features.float(), attention_mask.long()


# Re-score records from their normalized references and predictions so imported
# JSONL files are never trusted only because they contain aggregate numbers.
def score_records(records: list[dict]) -> dict:
    rescored = []
    for record in records:
        reference = normalize_ipa(record["reference"])
        # Earlier audit JSONL files used `prediction`; newly generated sweep
        # files use the more explicit `normalized_prediction` field.
        hypothesis_source = record.get(
            "normalized_prediction",
            record.get("prediction", record.get("raw_prediction")),
        )
        if hypothesis_source is None:
            raise KeyError("Prediction record has no recognized prediction-text field")
        hypothesis = normalize_ipa(hypothesis_source)
        substitutions, deletions, insertions = edit_counts(reference, hypothesis)
        updated = dict(record)
        updated.update({
            "reference": reference,
            "normalized_prediction": hypothesis,
            "S": substitutions,
            "D": deletions,
            "I": insertions,
        })
        rescored.append(updated)
    records[:] = rescored
    reference_codepoints = sum(len(row["reference"]) for row in records)
    substitutions = sum(int(row["S"]) for row in records)
    deletions = sum(int(row["D"]) for row in records)
    insertions = sum(int(row["I"]) for row in records)
    return {
        "clips": len(records),
        "reference_codepoints": reference_codepoints,
        "S": substitutions,
        "D": deletions,
        "I": insertions,
        "edit_count": substitutions + deletions + insertions,
        "corpus_cer_percent": 100 * (substitutions + deletions + insertions) / reference_codepoints,
        "eos_count": sum(bool(row.get("eos")) for row in records),
        "cap_without_eos": sum(bool(row.get("cap_without_eos")) for row in records),
        "replacement_character_outputs": sum("\ufffd" in row["normalized_prediction"] for row in records),
        "mean_prediction_reference_length_ratio": float(np.mean([
            len(row["normalized_prediction"]) / max(1, len(row["reference"])) for row in records
        ])),
    }


# Load newline-delimited private prediction records.
def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


## 6. Reuse and independently verify saved predictions when available

The prior audit contains already-generated predictions for penalties `1.00` and `1.05`. This block re-scores them from text, verifies all 732 frozen IDs and exact known aggregates, and makes them eligible for reuse. If the audit was not attached, the next block generates them again.


In [ ]:
# Build the exact expected validation-ID set without printing private IDs.
expected_ids = {row["sample_id"] for row in validation_rows}
cached_records = {}
cached_metrics = {}

if audit_root is not None:
    saved_files = {
        1.00: audit_root / "corrected_epoch_03_predictions_private.jsonl",
        1.05: audit_root / "epoch_03_repetition_penalty_1p05_predictions_private.jsonl",
    }
    known_aggregates = {
        1.00: {"S": 15940, "D": 4323, "I": 12669, "eos_count": 730, "cap_without_eos": 2},
        1.05: {"S": 14785, "D": 5236, "I": 11117, "eos_count": 732, "cap_without_eos": 0},
    }
    for penalty, path in saved_files.items():
        if not path.is_file():
            raise FileNotFoundError(f"Prior audit is missing its declared prediction file: {path.name}")
        records = load_jsonl(path)
        assert len(records) == EXPECTED_CLIPS
        assert {row["sample_id"] for row in records} == expected_ids
        metrics = score_records(records)
        assert metrics["reference_codepoints"] == EXPECTED_REFERENCE_CODEPOINTS
        for key, expected_value in known_aggregates[penalty].items():
            assert metrics[key] == expected_value, (
                f"Saved penalty {penalty:.2f} failed {key}: "
                f"{metrics[key]} != {expected_value}"
            )
        cached_records[penalty] = records
        cached_metrics[penalty] = metrics
    print("Verified and cached saved predictions for penalties 1.00 and 1.05.")
else:
    print("No prior audit attached; all five penalties will be generated from the checkpoint.")

if cached_metrics:
    display(pd.DataFrame([
        {"repetition_penalty": penalty, **metrics, "source": "verified saved predictions"}
        for penalty, metrics in sorted(cached_metrics.items())
    ]))


## 7. Deterministic batched generation

Each missing penalty is decoded over all 732 validation clips. The outer progress bar tracks penalty settings and the inner bar tracks 92 validation batches. On the previously measured T4 path, each full validation pass was roughly three minutes; current hardware and load can change that estimate.


In [ ]:
# Generate one full validation prediction set while changing only the declared
# repetition penalty. Every other GenerationConfig field is copied unchanged.
def generate_penalty(penalty: float) -> tuple[dict, list[dict]]:
    active_config = copy.deepcopy(generation_config)
    active_config.repetition_penalty = float(penalty)
    assert active_config.no_repeat_ngram_size == 0
    assert active_config.do_sample is False and active_config.num_beams == 1
    assert active_config.forced_decoder_ids is None
    assert active_config.task == "transcribe" and active_config.language is None

    records = []
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats(device)

    for begin in tqdm(
        range(0, len(validation_rows), VALIDATION_BATCH),
        desc=f"Penalty {penalty:.2f} validation",
        unit="batch",
        leave=True,
    ):
        rows = validation_rows[begin:begin + VALIDATION_BATCH]
        features, attention_mask = make_batch(rows)
        features = features.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)

        with torch.inference_mode():
            generated = model.generate(
                input_features=features,
                attention_mask=attention_mask,
                generation_config=active_config,
            )

        sequences = generated.sequences.detach().cpu().tolist()
        trimmed_ids = [
            sequence[:sequence.index(EOS_ID) + 1] if EOS_ID in sequence else sequence[:MAX_LENGTH]
            for sequence in sequences
        ]
        raw_predictions = processor.tokenizer.batch_decode(
            trimmed_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )

        for row, token_ids, raw_prediction in zip(rows, trimmed_ids, raw_predictions):
            reference = normalize_ipa(row["reference"])
            normalized_prediction = normalize_ipa(raw_prediction)
            substitutions, deletions, insertions = edit_counts(reference, normalized_prediction)
            records.append({
                "sample_id": row["sample_id"],
                "duration_seconds": float(row["duration_seconds"]),
                "reference": reference,
                "raw_prediction": raw_prediction,
                "normalized_prediction": normalized_prediction,
                "token_ids": token_ids,
                "output_length": len(token_ids),
                "eos": bool(token_ids and token_ids[-1] == EOS_ID),
                "cap_without_eos": bool(
                    len(token_ids) >= MAX_LENGTH and (not token_ids or token_ids[-1] != EOS_ID)
                ),
                "S": substitutions,
                "D": deletions,
                "I": insertions,
            })

        del features, attention_mask, generated, sequences

    torch.cuda.synchronize()
    elapsed_seconds = time.perf_counter() - started
    metrics = score_records(records)
    metrics.update({
        "elapsed_seconds": elapsed_seconds,
        "peak_gpu_allocated_mib": torch.cuda.max_memory_allocated(device) / 2**20,
        "repetition_penalty": float(penalty),
        "no_repeat_ngram_size": 0,
    })
    gc.collect()
    torch.cuda.empty_cache()
    return metrics, records


## 8. Run or reuse all five declared settings

Prediction-level JSONL outputs are private. Aggregates are accumulated in a comparison table. The notebook does not choose based on individual clips; selection uses the lowest full-corpus validation CER.


In [ ]:
# Run only missing settings while retaining a visible outer progress tracker.
all_records = {}
aggregate_rows = []

for penalty in tqdm(PENALTIES, desc="Repetition-penalty sweep", unit="setting"):
    if penalty in cached_records:
        records = cached_records[penalty]
        metrics = dict(cached_metrics[penalty])
        metrics.update({
            "repetition_penalty": float(penalty),
            "no_repeat_ngram_size": 0,
            "elapsed_seconds": None,
            "peak_gpu_allocated_mib": None,
            "source": "verified saved predictions",
        })
    else:
        metrics, records = generate_penalty(penalty)
        metrics["source"] = "generated in this notebook"

    assert len(records) == EXPECTED_CLIPS
    assert {row["sample_id"] for row in records} == expected_ids
    assert metrics["reference_codepoints"] == EXPECTED_REFERENCE_CODEPOINTS
    all_records[penalty] = records
    aggregate_rows.append(metrics)

    # Save every setting immediately so a later interruption does not erase a
    # completed validation pass.
    output_path = OUTPUT_ROOT / f"epoch03_repetition_penalty_{penalty:.2f}_predictions_private.jsonl"
    with output_path.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")

aggregate_table = pd.DataFrame(aggregate_rows).sort_values("repetition_penalty").reset_index(drop=True)
aggregate_table.to_csv(OUTPUT_ROOT / "repetition_penalty_sweep_metrics_private.csv", index=False)
(OUTPUT_ROOT / "repetition_penalty_sweep_metrics_private.json").write_text(
    json.dumps(aggregate_table.to_dict(orient="records"), indent=2, ensure_ascii=False) + "\n"
)

best_row = aggregate_table.loc[aggregate_table["corpus_cer_percent"].idxmin()]
print({
    "selected_by_validation_cer": float(best_row["repetition_penalty"]),
    "best_validation_cer_percent": float(best_row["corpus_cer_percent"]),
    "test_data_used": False,
})
display(aggregate_table[[
    "repetition_penalty", "corpus_cer_percent", "S", "D", "I",
    "eos_count", "cap_without_eos", "replacement_character_outputs",
    "mean_prediction_reference_length_ratio", "elapsed_seconds",
    "peak_gpu_allocated_mib", "source",
]])


## 9. Aggregate plots and insertion diagnostics

Eight complementary figures show the quality trade-off, edit composition, relative improvement, generation health, output length, selection decision, and the experimental flow. A private top-50 diagnostic file is created for later manual review without printing transcripts into the notebook output.


In [ ]:
# Use one consistent, color-blind-friendly visual system across all figures.
plt.rcParams.update({
    "figure.facecolor": "#f8fafc",
    "axes.facecolor": "#ffffff",
    "axes.edgecolor": "#cbd5e1",
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 11,
})
penalties = aggregate_table["repetition_penalty"].to_numpy(dtype=float)
labels = [f"{value:.2f}" for value in penalties]
colors = ["#2563eb", "#06b6d4", "#16a34a", "#f59e0b", "#ef4444"]
best_index = int(aggregate_table["corpus_cer_percent"].idxmin())
baseline_cer = float(
    aggregate_table.loc[aggregate_table["repetition_penalty"].eq(1.00), "corpus_cer_percent"].iloc[0]
)


# Save a figure, show it inline, and close it to avoid notebook memory growth.
def save_show(fig, filename: str) -> Path:
    path = PLOTS_ROOT / filename
    fig.savefig(path, dpi=190, bbox_inches="tight", facecolor=fig.get_facecolor())
    display(fig)
    plt.close(fig)
    return path


# Figure 1: direct CER comparison with the validation-selected point highlighted.
fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(labels, aggregate_table["corpus_cer_percent"], color=colors, edgecolor="white", linewidth=1.5)
bars[best_index].set_edgecolor("#052e16")
bars[best_index].set_linewidth(3)
for bar, value in zip(bars, aggregate_table["corpus_cer_percent"]):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 0.25, f"{value:.2f}%", ha="center", fontweight="bold")
ax.axhline(baseline_cer, color="#475569", linestyle="--", linewidth=1.5, label="Penalty 1.00 baseline")
ax.set(title="Which repetition penalty gives the lowest validation CER?", xlabel="Repetition penalty", ylabel="Corpus CER (%)")
ax.legend(loc="upper right")
save_show(fig, "01_cer_comparison.png")


# Figure 2: absolute S/D/I counts show how each setting changes the error mechanism.
fig, ax = plt.subplots(figsize=(11, 6))
width = 0.25
x = np.arange(len(labels))
ax.bar(x - width, aggregate_table["S"], width, label="Substitutions", color="#8b5cf6")
ax.bar(x, aggregate_table["D"], width, label="Deletions", color="#f59e0b")
ax.bar(x + width, aggregate_table["I"], width, label="Insertions", color="#ec4899")
ax.set_xticks(x, labels)
ax.set(title="Character-error counts by penalty", xlabel="Repetition penalty", ylabel="Edit operations")
ax.legend(ncol=3)
save_show(fig, "02_edit_counts_grouped.png")


# Figure 3: normalized error composition distinguishes lower totals from a mere
# trade of insertions for deletions.
fig, ax = plt.subplots(figsize=(10, 5.5))
edit_matrix = aggregate_table[["S", "D", "I"]].to_numpy(dtype=float)
edit_percent = 100 * edit_matrix / edit_matrix.sum(axis=1, keepdims=True)
bottom = np.zeros(len(labels))
for column, name, color in zip(edit_percent.T, ["Substitutions", "Deletions", "Insertions"], ["#8b5cf6", "#f59e0b", "#ec4899"]):
    ax.bar(labels, column, bottom=bottom, label=name, color=color)
    bottom += column
ax.set(title="What share of the remaining errors is S, D, or I?", xlabel="Repetition penalty", ylabel="Share of all edits (%)", ylim=(0, 100))
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.14))
save_show(fig, "03_error_composition_percent.png")


# Figure 4: positive bars are absolute CER-point improvements over no penalty.
fig, ax = plt.subplots(figsize=(10, 5.5))
improvement = baseline_cer - aggregate_table["corpus_cer_percent"].to_numpy(dtype=float)
delta_colors = ["#16a34a" if value > 0 else "#dc2626" if value < 0 else "#64748b" for value in improvement]
bars = ax.bar(labels, improvement, color=delta_colors)
ax.axhline(0, color="#0f172a", linewidth=1)
for bar, value in zip(bars, improvement):
    ax.text(bar.get_x() + bar.get_width() / 2, value + (0.08 if value >= 0 else -0.18), f"{value:+.2f}", ha="center", va="bottom" if value >= 0 else "top", fontweight="bold")
ax.set(title="CER-point change relative to penalty 1.00", xlabel="Repetition penalty", ylabel="CER points improved (higher is better)")
save_show(fig, "04_cer_improvement_from_baseline.png")


# Figure 5: EOS, cap, and replacement-character health indicators.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
health_specs = [
    ("eos_count", "EOS-terminated outputs", "#14b8a6"),
    ("cap_without_eos", "Cap hits without EOS", "#ef4444"),
    ("replacement_character_outputs", "U+FFFD outputs", "#f97316"),
]
for ax, (column, title, color) in zip(axes, health_specs):
    values = aggregate_table[column].to_numpy(dtype=float)
    ax.bar(labels, values, color=color, alpha=0.88)
    for index, value in enumerate(values):
        ax.text(index, value + max(0.15, values.max() * 0.015), f"{int(value)}", ha="center", fontweight="bold")
    ax.set(title=title, xlabel="Penalty", ylabel="Clips")
fig.suptitle("Generation-health comparison across all 732 validation clips", fontsize=14, fontweight="bold")
fig.tight_layout()
save_show(fig, "05_generation_health.png")


# Figure 6: output/reference length behavior is a direct over-generation clue.
fig, ax = plt.subplots(figsize=(10, 5.5))
length_ratio = aggregate_table["mean_prediction_reference_length_ratio"].to_numpy(dtype=float)
ax.plot(labels, length_ratio, color="#7c3aed", marker="o", linewidth=3, markersize=9)
ax.axhline(1.0, color="#334155", linestyle="--", label="equal prediction/reference length")
for index, value in enumerate(length_ratio):
    ax.text(index, value + 0.015, f"{value:.3f}", ha="center", fontweight="bold")
ax.set(title="Does the decoder still over-generate?", xlabel="Repetition penalty", ylabel="Mean prediction/reference length ratio")
ax.legend()
save_show(fig, "06_output_length_ratio.png")


# Figure 7: a compact decision dashboard combines CER, insertions, deletions,
# and cap hits without hiding the trade-off behind one score.
fig = plt.figure(figsize=(14, 8))
grid = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.25)
dashboard_specs = [
    ("corpus_cer_percent", "CER (%)", "#2563eb"),
    ("I", "Insertions", "#ec4899"),
    ("D", "Deletions", "#f59e0b"),
    ("cap_without_eos", "Cap hits without EOS", "#ef4444"),
]
for position, (column, title, color) in enumerate(dashboard_specs):
    ax = fig.add_subplot(grid[position // 2, position % 2])
    ax.plot(labels, aggregate_table[column], marker="o", linewidth=2.5, color=color)
    ax.axvspan(best_index - 0.18, best_index + 0.18, color="#bbf7d0", alpha=0.65)
    ax.set(title=title, xlabel="Penalty")
fig.suptitle(f"Selection dashboard — current minimum CER at penalty {float(best_row['repetition_penalty']):.2f}", fontsize=16, fontweight="bold")
save_show(fig, "07_selection_dashboard.png")


# Figure 8: a colorful process illustration explains what is fixed and what is
# varied, helping readers interpret this as decoding selection rather than training.
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 4.5)
ax.axis("off")
steps = [
    (1.5, "Frozen validation\n732 clips", "#dbeafe", "#2563eb"),
    (4.3, "F1 Epoch 3\nfixed weights", "#ede9fe", "#7c3aed"),
    (7.1, "Sweep only\nrepetition penalty", "#fef3c7", "#d97706"),
    (9.9, "Same CER\ncontract", "#cffafe", "#0891b2"),
    (12.5, "Select on\nvalidation only", "#dcfce7", "#16a34a"),
]
for index, (xpos, text_value, face, edge) in enumerate(steps):
    ax.text(xpos, 2.25, text_value, ha="center", va="center", fontsize=12, fontweight="bold",
            bbox={"boxstyle": "round,pad=0.7", "facecolor": face, "edgecolor": edge, "linewidth": 2})
    if index < len(steps) - 1:
        ax.annotate("", xy=(steps[index + 1][0] - 0.9, 2.25), xytext=(xpos + 0.9, 2.25),
                    arrowprops={"arrowstyle": "-|>", "color": "#475569", "linewidth": 2})
ax.text(7, 4.05, "F1-RP experiment flow", ha="center", fontsize=17, fontweight="bold", color="#0f172a")
ax.text(7, 0.35, "Test set remains sealed • no retraining • no normalization or CER changes", ha="center", fontsize=11, color="#475569")
save_show(fig, "08_experiment_flow.png")

# Rank the selected setting's most insertion-heavy examples for a separate,
# private manual review. References and predictions are saved but not displayed.
selected_penalty = float(best_row["repetition_penalty"])
selected_records = all_records[selected_penalty]
diagnostics = []
for record in selected_records:
    row = dict(record)
    row["prediction_reference_length_ratio"] = (
        len(row["normalized_prediction"]) / max(1, len(row["reference"]))
    )
    row["clip_cer_percent"] = 100 * (row["S"] + row["D"] + row["I"]) / max(1, len(row["reference"]))
    diagnostics.append(row)
diagnostics.sort(
    key=lambda row: (row["I"], row["prediction_reference_length_ratio"], row["clip_cer_percent"]),
    reverse=True,
)
top_diagnostics = diagnostics[:50]
(OUTPUT_ROOT / "selected_penalty_top50_insertion_diagnostics_private.json").write_text(
    json.dumps(top_diagnostics, indent=2, ensure_ascii=False) + "\n"
)

print({
    "selected_penalty": selected_penalty,
    "diagnostic_examples_saved": len(top_diagnostics),
    "diagnostic_transcripts_printed": False,
    "plots_saved": len(list(PLOTS_ROOT.glob("*.png"))),
    "plots_directory": str(PLOTS_ROOT),
})


## 10. Save downloadable ZIPs for all results and all plots

The **complete-results ZIP** includes configuration, aggregate metrics, diagnostics, all five prediction JSONL files, and every plot. The separate **plots ZIP** contains only the eight PNG figures and their checksums. The results ZIP contains private IDs, references, predictions, and token sequences for newly generated settings, so do not commit it to GitHub.


In [ ]:
# Record package versions and input fingerprints needed to reproduce the sweep.
run_summary = {
    "experiment": "F1-RP",
    "status": "completed",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "checkpoint": "F1 Epoch 3 selected checkpoint",
    "checkpoint_model_sha256": sha256(checkpoint_model_root / "model.safetensors"),
    "validation_manifest_sha256": bundle_record["manifest_sha256"],
    "validation_clips": EXPECTED_CLIPS,
    "reference_codepoints": EXPECTED_REFERENCE_CODEPOINTS,
    "penalties": PENALTIES,
    "selected_penalty_by_validation_cer": float(best_row["repetition_penalty"]),
    "selected_validation_cer_percent": float(best_row["corpus_cer_percent"]),
    "test_data_used": False,
    "packages": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "tokenizers": version("tokenizers"),
        "soundfile": version("soundfile"),
        "safetensors": version("safetensors"),
    },
    "gpu": torch.cuda.get_device_name(0),
}
(OUTPUT_ROOT / "run_summary_private.json").write_text(
    json.dumps(run_summary, indent=2, ensure_ascii=False) + "\n"
)

# Create hashes after all result and plot files have been finalized.
result_files = sorted(path for path in OUTPUT_ROOT.rglob("*") if path.is_file())
with (OUTPUT_ROOT / "SHA256SUMS.txt").open("w", encoding="utf-8") as stream:
    for path in result_files:
        stream.write(f"{sha256(path)}  {path.relative_to(OUTPUT_ROOT)}\n")
plot_files = sorted(PLOTS_ROOT.glob("*.png"))
assert len(plot_files) == 8, f"Expected eight plots, found {len(plot_files)}"
with (PLOTS_ROOT / "SHA256SUMS.txt").open("w", encoding="utf-8") as stream:
    for path in plot_files:
        stream.write(f"{sha256(path)}  {path.name}\n")

# Package results and plots together for complete private preservation.
results_zip_path = Path("/kaggle/working/phase7_F1_repetition_penalty_sweep_all_results_private.zip")
with zipfile.ZipFile(results_zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in tqdm(sorted(OUTPUT_ROOT.rglob("*")), desc="Building result ZIP", unit="file"):
        if path.is_file():
            archive.write(path, Path("phase7_F1_repetition_penalty_sweep/results") / path.relative_to(OUTPUT_ROOT))
    for path in sorted(PLOTS_ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, Path("phase7_F1_repetition_penalty_sweep/plots") / path.relative_to(PLOTS_ROOT))

# Package a smaller plot-only archive for convenient download and sharing.
plots_zip_path = Path("/kaggle/working/phase7_F1_repetition_penalty_sweep_all_plots.zip")
with zipfile.ZipFile(plots_zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in tqdm(sorted(PLOTS_ROOT.rglob("*")), desc="Building plots ZIP", unit="file"):
        if path.is_file():
            archive.write(path, Path("phase7_F1_repetition_penalty_sweep_plots") / path.relative_to(PLOTS_ROOT))

with zipfile.ZipFile(results_zip_path) as archive:
    archived_result_files = [name for name in archive.namelist() if not name.endswith("/")]
with zipfile.ZipFile(plots_zip_path) as archive:
    archived_plot_files = [name for name in archive.namelist() if not name.endswith("/")]

print({
    "all_results_zip": str(results_zip_path),
    "all_results_size_mib": round(results_zip_path.stat().st_size / 2**20, 2),
    "all_results_files": len(archived_result_files),
    "all_results_sha256": sha256(results_zip_path),
    "all_plots_zip": str(plots_zip_path),
    "all_plots_size_mib": round(plots_zip_path.stat().st_size / 2**20, 2),
    "all_plots_files": len(archived_plot_files),
    "all_plots_sha256": sha256(plots_zip_path),
    "private_output": True,
})
display(FileLink(results_zip_path))
display(FileLink(plots_zip_path))


## Interpretation and gate

The penalty with the lowest full-corpus validation CER is the **validation-selected F1 decoding candidate**. It is not a test result and does not by itself prove better unseen-test performance.

After downloading the ZIP and the executed notebook:

1. Review the private top-50 insertion diagnostics.
2. Record whether the selected penalty mainly reduced repeated/hallucinated continuation or merely traded insertions for deletions.
3. Keep the 683-clip test set sealed.
4. Start the matched L1 LoRA experiment independently from the original pinned pretrained checkpoint.
5. Compare F1 and L1 on validation, lock one checkpoint and one decoding policy, and only then run the final sealed test once.
